In [ ]:
import os
import scanpy as sc
import scipy
import pandas as pd
import matplotlib.pyplot as plt
import random
from LingerGRN.pseudo_bulk import *
from LingerGRN.preprocess import *
import LingerGRN.LINGER_tr as LINGER_tr
import LingerGRN.LL_net as LL_net
from LingerGRN.TF_activity import *

sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(5, 5), facecolor='white')
sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()

plt.rcParams['font.family']=['Dejavu serif']
plt.rcParams['figure.dpi'] = 100
plt.rcParams['pdf.fonttype'] = 'truetype'

root = path_to_wd
os.chdir(root)

In [ ]:
dir_path = f'{root}Reproducibility/Results/LINGER/TREKKER/'
os.makedirs(dir_path, exist_ok = True)

outdir=f"{dir_path}output/"
os.makedirs(outdir, exist_ok = True)

datadir=f"{dir_path}data/"
os.makedirs(datadir, exist_ok = True)

## Prepare the input data

In [ ]:
adata = sc.read("Reproducibility/Data/UC_TREKKER_RNA_ATAC_Epithelial.h5ad")

In [ ]:
adata.obs["label"] = adata.obs["epi_phenotype"]
adata.obs['barcode'] = adata.obs_names

In [ ]:
label = pd.DataFrame({'barcode_use': adata.obs_names,
                      'label': adata.obs.label
                     })

In [ ]:
## Remove samples <100 cells
adata = adata[adata.obs['sample'].isin(['P03'])==False]

In [ ]:
# Fig.4A
sc.set_figure_params(figsize=(4, 4)) 
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.umap(
    adata,
    color=['label','sample'],
    frameon=False,
    #legend_loc="on data",
    legend_fontsize=7,
    show = True
)

### Remove low counts cells and genes

In [ ]:
adata_RNA  = adata[:, adata.var.feature_types == "Gene Expression"].copy() 
adata_ATAC = adata[:, adata.var.feature_types == "Peaks"].copy()

sc.pp.filter_cells(adata_RNA, min_genes=400)
sc.pp.filter_genes(adata_RNA, min_cells=3)
sc.pp.filter_cells(adata_ATAC, min_genes=400)
sc.pp.filter_genes(adata_ATAC, min_cells=3)

selected_barcode=list(set(adata_RNA.obs['barcode'].values)&set(adata_ATAC.obs['barcode'].values))

In [ ]:
adata_RNA = adata_RNA[selected_barcode,:].copy()
adata_ATAC = adata_ATAC[selected_barcode,:].copy()

import re
adata_RNA.var['gene_ids'] = adata_RNA.var_names
adata_ATAC.var.index = adata_ATAC.var.index.to_series().apply(lambda x: re.sub(r'^([^-\n]+)-', r'\1:', x))
adata_ATAC.var['gene_ids'] = adata_ATAC.var_names

In [ ]:
adata_RNA.raw = adata_RNA
adata_ATAC.raw = adata_ATAC

## Generate the pseudo-bulk/metacell 

In [ ]:
def pseudo_bulk_modified(adata_RNA,adata_ATAC,singlepseudobulk):
    K = 20
    sc.pp.normalize_total(adata_RNA, target_sum=1e4)
    sc.pp.log1p(adata_RNA)
    adata_RNA.raw=adata_RNA 
    sc.pp.highly_variable_genes(adata_RNA, min_mean=0.0125, max_mean=3, min_disp=0.5)
    adata_RNA = adata_RNA[:, adata_RNA.var.highly_variable]
    sc.pp.scale(adata_RNA, max_value=10)
    sc.tl.pca(adata_RNA, n_comps=15,svd_solver="arpack")
    pca_RNA=adata_RNA.obsm['X_pca']
    sc.pp.log1p(adata_ATAC)
    adata_ATAC.raw=adata_ATAC
    sc.pp.highly_variable_genes(adata_ATAC, min_mean=0.0125, max_mean=3, min_disp=0.5)
    adata_ATAC = adata_ATAC[:, adata_ATAC.var.highly_variable]
    sc.pp.scale(adata_ATAC, max_value=10, zero_center=True)
    sc.tl.pca(adata_ATAC, n_comps=15,svd_solver="arpack")
    pca_ATAC=adata_ATAC.obsm['X_pca']
    pca = np.concatenate((pca_RNA,pca_ATAC), axis=1)
    adata_RNA.obsm['pca']=pca
    adata_ATAC.obsm['pca']=pca
    sc.pp.neighbors(adata_RNA, n_neighbors=K, n_pcs=30,use_rep='pca')
    connectivities=(adata_RNA.obsp['distances']>0)
    import random
    label=pd.DataFrame(adata_RNA.obs['label'])
    label.columns=['label']
    label.index=adata_RNA.obs['barcode'].tolist()
    #label=label['label'].values
    cluster=list(set(label['label'].values))
    allindex=[]
    # np.random.seed(42)  # Set seed for reproducibility
    random.seed(42)  # Set seed for reproducibility
    for i in range(len(cluster)):
        temp=label[label['label']==cluster[i]].index
        N = len(temp) # Total number of elements
        if N>=10:
            #m = int(np.floor(np.sqrt(N)))+1  # Number of elements to sample
            m = int(np.floor(N ** (1/3))) + 1
            if singlepseudobulk>0:
                m=2             ## m=1だとelementが少なすぎる
            sampled_elements = random.sample(range(N), m)
            temp=temp[sampled_elements]
            allindex=allindex+temp.tolist()
    connectivities=pd.DataFrame(connectivities.toarray(),index=adata_RNA.obs['barcode'].tolist())
    connectivities=connectivities.loc[allindex].values
    A=(connectivities @ adata_RNA.raw.X.toarray())
    TG_filter1=A/(K-1)
    RE_filter1=(connectivities @ adata_ATAC.raw.X.toarray())/(K-1)
    TG_filter1=pd.DataFrame(TG_filter1.T,columns=allindex,index=adata_RNA.raw.var['gene_ids'].tolist())
    RE_filter1=pd.DataFrame(RE_filter1.T,columns=allindex,index=adata_ATAC.raw.var['gene_ids'].tolist())
    return TG_filter1,RE_filter1

In [ ]:
samplelist=list(set(adata_ATAC.obs['sample'].values)) # sample is generated from cell barcode 
tempsample=samplelist[0]
TG_pseudobulk=pd.DataFrame([])
RE_pseudobulk=pd.DataFrame([])
singlepseudobulk = (adata_RNA.obs['sample'].unique().shape[0]*adata_RNA.obs['sample'].unique().shape[0]>100)
for tempsample in samplelist:
    adata_RNAtemp=adata_RNA[adata_RNA.obs['sample']==tempsample]
    adata_ATACtemp=adata_ATAC[adata_ATAC.obs['sample']==tempsample]
    TG_pseudobulk_temp,RE_pseudobulk_temp=pseudo_bulk_modified(adata_RNAtemp,adata_ATACtemp,singlepseudobulk)                
    TG_pseudobulk=pd.concat([TG_pseudobulk, TG_pseudobulk_temp], axis=1)
    RE_pseudobulk=pd.concat([RE_pseudobulk, RE_pseudobulk_temp], axis=1)
    RE_pseudobulk[RE_pseudobulk > 100] = 100

In [ ]:
adata_ATAC.write(f'{datadir}adata_ATAC.h5ad')
adata_RNA.write(f'{datadir}adata_RNA.h5ad')

In [ ]:
TG_pseudobulk=TG_pseudobulk.fillna(0)
RE_pseudobulk=RE_pseudobulk.fillna(0)

TG_pseudobulk.to_csv(f'{datadir}TG_pseudobulk.tsv')
RE_pseudobulk.to_csv(f'{datadir}RE_pseudobulk.tsv')

adata_ATAC.var.loc[RE_pseudobulk.index,'gene_ids'].to_csv(f'{datadir}Peaks.txt',header=None,index=None)

## Overlap the region with general GRN

In [ ]:
os.chdir(dir_path)

genome='hg38'
method = 'LINGER'
Datadir='~/reference/LINGER/' 
GRNdir='~/reference/LINGER/data_bulk/'

preprocess(TG_pseudobulk,RE_pseudobulk,GRNdir,genome,method,outdir)

## Train for the LINGER model

In [ ]:
activef ='ReLU'
species = 'Human'

LINGER_tr.training(GRNdir,method,outdir,activef,species)

## Cell population gene regulatory network

In [ ]:
genome = 'hg38'
network = 'cell population'

# TF binding potential
LL_net.TF_RE_binding(GRNdir,adata_RNA,adata_ATAC,genome,method,outdir)
# cis-regulatory network
LL_net.cis_reg(GRNdir,adata_RNA,adata_ATAC,genome,method,outdir)
# trans-regulatory network
LL_net.trans_reg(GRNdir,method,outdir,genome)

### Calculate TF activity

In [ ]:
os.chdir(root)
adata = sc.read("Reproducibility/Data/UC_TREKKER_RNA_ATAC_Epithelial.h5ad")
adata.obs["label"] = adata.obs["epi_phenotype"]
adata.obs['barcode'] = adata.obs_names
adata_RNA  = adata[:, adata.var.feature_types == "Gene Expression"].copy() 
adata_RNA.var['gene_ids'] = adata_RNA.var_names

genome = 'hg38'
network = 'cell population'
dir_path = f'{path_to_wd}/Reproducibility/Results/LINGER/TREKKER/'
os.chdir(dir_path)
outdir=f"{dir_path}output/"

TF_activity = regulon(outdir,adata_RNA,GRNdir,network,genome)
TF_activity = TF_activity.dropna(how='all').dropna(how='all', axis=1)
TF_activity.to_csv(f"{outdir}cell_population_TF_activity.txt", sep='\t')

In [ ]:
TF_use = np.intersect1d(adata_RNA.var_names, TF_activity.index)

# RNA
sc.pp.normalize_total(adata_RNA)
sc.pp.log1p(adata_RNA)
sc.pp.scale(adata_RNA, max_value=10)

# TF_activity
adata_TF = sc.AnnData(TF_activity.T.loc[adata_RNA.obs_names,:].copy(), obs=adata_RNA.obs)
sc.pp.scale(adata_TF, max_value=10)

exp_df = pd.DataFrame(adata_RNA[:, TF_use].X, index=adata_RNA.obs_names, columns=TF_use)
TF_df  = pd.DataFrame(adata_TF[:, TF_use].X,  index=adata_TF.obs_names, columns=TF_use)

TF_df.to_csv(f"{outdir}cell_population_TF_activity_zscore.txt",sep='\t')

In [ ]:
for tmp_celltype in adata_RNA.obs['label'].unique():
    CB_use = adata_RNA[adata_RNA.obs['label'].isin([tmp_celltype])].obs_names
    exp_df_tmp = exp_df.loc[CB_use,:].copy().T
    TF_df_tmp = TF_df.loc[CB_use,:].copy().T
    ## average
    exp_TF_df = pd.DataFrame(index = exp_df_tmp.index, columns = ['EXP', 'TF'])
    exp_TF_df.loc[:,'EXP'] = exp_df_tmp.mean(axis='columns')
    exp_TF_df.loc[:,'TF'] = TF_df_tmp.mean(axis='columns')
    exp_TF_df.to_csv(f"{outdir}cell_population_exp_TF_activity_zscore_{tmp_celltype}.txt",sep='\t')

- P02

In [ ]:
root = path_to_wd
os.chdir(root)

adata = sc.read("Reproducibility/Data/UC_TREKKER_RNA_ATAC_Epithelial.h5ad")
adata.obs["label"] = adata.obs["epi_phenotype"]
adata.obs['barcode'] = adata.obs_names
adata_RNA  = adata[:, adata.var.feature_types == "Gene Expression"].copy() 
adata_RNA.var['gene_ids'] = adata_RNA.var_names

dir_path = 'Reproducibility/Results/LINGER/TREKKER/'
outdir=f"{dir_path}output/"

TF_activity = pd.read_csv(outdir+'cell_population_TF_activity.txt',sep='\t', index_col=0)
TF_use = np.intersect1d(adata_RNA.var_names, TF_activity.index)

In [ ]:
adata_tmp = adata_RNA[adata_RNA.obs['clone'].isin(['P02_clone_1','P02_clone_2'])].copy()
sc.pp.normalize_total(adata_tmp)
sc.pp.log1p(adata_tmp)
sc.pp.scale(adata_tmp, max_value=10)

# TF_activity
adata_TF = sc.AnnData(TF_activity.T.loc[adata_tmp.obs_names,:].copy(), obs=adata_tmp.obs)
sc.pp.scale(adata_TF, max_value=10)

TF_df  = pd.DataFrame(adata_TF[:, TF_use].X,  index=adata_TF.obs_names, columns=TF_use)
TF_df.to_csv(f'{outdir}cell_population_TF_activity_zscore_P02.txt',sep='\t')

- P06

In [ ]:
adata_tmp = adata_RNA[adata_RNA.obs['clone'].isin(['P06_clone_1','P06_clone_2','P06_clone_3','P06_clone_4'])].copy()
sc.pp.normalize_total(adata_tmp)
sc.pp.log1p(adata_tmp)
sc.pp.scale(adata_tmp, max_value=10)

# TF_activity
adata_TF = sc.AnnData(TF_activity.T.loc[adata_tmp.obs_names,:].copy(), obs=adata_tmp.obs)
sc.pp.scale(adata_TF, max_value=10)

TF_df  = pd.DataFrame(adata_TF[:, TF_use].X,  index=adata_TF.obs_names, columns=TF_use)
TF_df.to_csv(f'{outdir}cell_population_TF_activity_zscore_P06.txt',sep='\t')